1.Import & Setup

In [1]:
import pandas as pd
import numpy as np
import json, gzip, os, sys, re
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import sqlite3
import gc
import pyarrow.parquet as pq
from concurrent.futures import ThreadPoolExecutor
from queue import Queue
import threading

2. Path

In [2]:
ROOT_DIR      = Path().resolve().parent
REVIEW_PATH   = ROOT_DIR / "data" / "raw" / "Clothing_Shoes_and_Jewelry.jsonl.gz"
META_PATH     = ROOT_DIR / "data" / "raw" / "meta_Clothing_Shoes_and_Jewelry.jsonl.gz"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
SAMPLE_DIR    = ROOT_DIR / "data" / "sample"
FIGURES_DIR   = ROOT_DIR / "outputs" / "figures"
FINAL_META_CSV = PROCESSED_DIR / "meta_clean_all.csv"
FINAL_REVIEW_CSV = PROCESSED_DIR / "review_clean_all.csv"
CHUNK_SIZE = 500000

3. Hàm tiền xử lý review

In [3]:
def preprocess_review(df):
    df = df.copy()
    log = {}  # Ghi lại số dòng sau mỗi bước
    log['1_raw'] = len(df)

    # ----------------------------------------------------------
    # BƯỚC A: Chọn cột cần thiết
    # Chỉ giữ các cột có ý nghĩa cho phân tích
    # ----------------------------------------------------------
    needed = ['rating', 'title', 'user_id','text',
              'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
    existing_cols = [c for c in needed if c in df.columns]
    df = df[existing_cols]
    print(f"A. Chọn cột: giữ lại {existing_cols}")

    # ----------------------------------------------------------
    # BƯỚC B: Xử lý missing values
    # rating và title là BẮT BUỘC → drop nếu thiếu
    # các cột khác → fill giá trị mặc định
    # ----------------------------------------------------------
    df = df.dropna(subset=['rating', 'title'])
    df['helpful_vote']      = df.get('helpful_vote', pd.Series(0)).fillna(0)
    df['verified_purchase'] = df.get('verified_purchase', pd.Series(False)).fillna(False)
    log['2_drop_na'] = len(df)
    print(f"B. Drop NA  : {log['1_raw']:,} → {log['2_drop_na']:,} (-{log['1_raw']-log['2_drop_na']:,})")

    # ----------------------------------------------------------
    # BƯỚC C: Chuẩn hóa kiểu dữ liệu
    # ----------------------------------------------------------
    df['rating']        = pd.to_numeric(df['rating'], errors='coerce')
    df['helpful_vote']  = pd.to_numeric(df['helpful_vote'], errors='coerce').fillna(0).astype(int)
    df = df.dropna(subset=['rating'])  # drop nếu rating không parse được
    df['rating']        = df['rating'].astype(float)
    log['3_dtype'] = len(df)
    print(f"C. Dtype    : {log['2_drop_na']:,} → {log['3_dtype']:,}")

    # ----------------------------------------------------------
    # BƯỚC D: Chuyển timestamp → datetime
    # timestamp gốc là Unix milliseconds (số ms từ 1970)
    # ----------------------------------------------------------
    if 'timestamp' in df.columns:
        df['date']  = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
        df['year']  = df['date'].dt.year
        df['month'] = df['date'].dt.month
        print(f"D. Timestamp: range {df['year'].min():.0f} - {df['year'].max():.0f}")

    # ----------------------------------------------------------
    # BƯỚC E: Tạo nhãn Sentiment từ Rating
    # 4-5 sao → positive | 3 sao → neutral | 1-2 sao → negative
    # Đây là SUPERVISED LABEL cho bài toán Sentiment Analysis
    # ----------------------------------------------------------
    def to_sentiment(r):
        if r >= 4:   return 'positive'
        elif r == 3: return 'neutral'
        else:        return 'negative'

    df['sentiment']       = df['rating'].apply(to_sentiment)
    df['sentiment_score'] = df['rating'].apply(lambda r: 1 if r >= 4 else (0 if r == 3 else -1))
    print(f"E. Sentiment: {df['sentiment'].value_counts().to_dict()}")

    # ----------------------------------------------------------
    # BƯỚC F: Lọc review quá ngắn (noise)
    # Review < 10 ký tự không có giá trị phân tích NLP
    # ----------------------------------------------------------
    if 'text' in df.columns:
        df['text_length'] = df['text'].astype(str).str.len()
        df = df[df['text_length'] >= 10] # Giả sử MIN_TEXT_LENGTH = 10
        
        # QUAN TRỌNG: Xóa cột 'text' và 'text_length' sau khi lọc xong
        df = df.drop(columns=['text', 'text_length'])

    # ----------------------------------------------------------
    # BƯỚC G: Loại bỏ duplicate
    # Cùng 1 user review cùng 1 sản phẩm → chỉ giữ lần đầu
    # ----------------------------------------------------------
    before = len(df)
    df = df.drop_duplicates(subset=['user_id', 'parent_asin'], keep='first')
    log['5_dedup'] = len(df)
    print(f"G. Duplicate: {before:,} → {log['5_dedup']:,} (-{before-log['5_dedup']:,} duplicates)")

    # ----------------------------------------------------------
    # BƯỚC H: Reset index
    # ----------------------------------------------------------
    df = df.reset_index(drop=True)

    print(f"\n✅ Kết quả cuối: {len(df):,} reviews")
    return df, log



4. Hàm tiền xử lý meta

In [4]:
def preprocess_meta(df):
    df = df.copy()
    print(f"Raw shape: {df.shape}")

    # ----------------------------------------------------------
    # BƯỚC A: Chọn cột cần thiết
    # ----------------------------------------------------------
    needed = ['parent_asin', 'title', 'description',
              'categories', 'average_rating', 'rating_number',
              'store', 'main_category','images','price']
    existing_cols = [c for c in needed if c in df.columns]
    df = df[existing_cols]
    print(f"A. Giữ cột: {existing_cols}")

    # ----------------------------------------------------------
    # BƯỚC B: Drop sản phẩm không có ID hoặc tên
    # ----------------------------------------------------------
    before = len(df)
    df = df.dropna(subset=['parent_asin', 'title'])
    df = df.drop_duplicates(subset=['parent_asin'], keep='first')
    print(f"B. Drop NA/dup: {before:,} → {len(df):,}")

    # ----------------------------------------------------------
    # BƯỚC C: Xử lý Price
    # Giá có thể là '$29.99' hoặc '29.99' → cần chuẩn hóa
    # Lọc outlier: giá <= 0 hoặc > $10,000 là bất thường
    # ----------------------------------------------------------
    if 'price' in df.columns:
        df['price'] = df['price'].astype(str).str.replace(r'[^\d.]', '', regex=True)
        df['price'] = pd.to_numeric(df['price'], errors='coerce')
        
        # Lọc: Chỉ giữ sản phẩm có giá hợp lý (ví dụ > 0)
        df = df[df['price'] > 0] 
        
        # QUAN TRỌNG: Xóa cột 'price' sau khi lọc
        df = df.drop(columns=['price'])

    # ----------------------------------------------------------
    # BƯỚC D: Xử lý Description (list → string)
    # description gốc là list các đoạn văn
    # ----------------------------------------------------------
    if 'description' in df.columns:
        df['description'] = df['description'].apply(
            lambda x: ' '.join(x) if isinstance(x, list) else str(x) if pd.notna(x) else ''
        )
        print(f"D. Description: đã chuyển list → string")

    # ----------------------------------------------------------
    # BƯỚC E: Xử lý Categories
    # categories là list lồng nhau → lấy level 1 làm main_category
    # ----------------------------------------------------------
    if 'categories' in df.columns:
        def extract_category(cats):
            if isinstance(cats, list) and len(cats) > 0:
                inner = cats[0] if isinstance(cats[0], list) else cats
                return inner[0] if len(inner) > 0 else 'Unknown'
            return 'Unknown'
        df['main_category'] = df['categories'].apply(extract_category)
        print(f"E. Categories top 5: {df['main_category'].value_counts().head().to_dict()}")

    df = df.reset_index(drop=True)
    print(f"\n✅ Meta kết quả: {df.shape}")
    return df

5. Đọc+ tiền xử lý và lưu file review

In [5]:
CHUNK_SIZE = 500_000
N_WORKERS  = 4

if os.path.exists(FINAL_REVIEW_CSV):
    os.remove(FINAL_REVIEW_CSV)
    print("🗑️ Đã dọn dẹp file Review cũ. Bắt đầu tạo file mới hoàn toàn...")
def process_and_save_review(chunk_data, file_exists_flag, write_lock):
    """Preprocess 1 chunk rồi ghi CSV — chạy trong luồng con."""
    df_temp, _ = preprocess_review(pd.DataFrame(chunk_data))
    with write_lock:
        df_temp.to_csv(
            FINAL_REVIEW_CSV,
            mode='a',
            index=False,
            header=not file_exists_flag[0]
        )
        file_exists_flag[0] = True  # Đánh dấu file đã tồn tại
    del df_temp
    gc.collect()
 
print(f"📦 Đang xử lý Review theo cụm {CHUNK_SIZE:,} (đa luồng)...")
 
file_exists_flag = [False]  # Dùng list để truyền tham chiếu vào luồng con
write_lock       = threading.Lock()
current_chunk    = []
 
with gzip.open(REVIEW_PATH, 'rt', encoding='utf-8') as f, \
     ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
 
    futures = []
    for line in tqdm(f, desc="Streaming Review"):
        try:
            current_chunk.append(json.loads(line.strip()))
        except:
            continue
 
        if len(current_chunk) == CHUNK_SIZE:
            # Nộp chunk vào luồng con ngay, không chờ xử lý xong
            futures.append(executor.submit(
                process_and_save_review,
                current_chunk.copy(),
                file_exists_flag,
                write_lock
            ))
            current_chunk = []
 
    # Xử lý phần dư cuối file
    if current_chunk:
        futures.append(executor.submit(
            process_and_save_review,
            current_chunk.copy(),
            file_exists_flag,
            write_lock
        ))
 
    # Chờ tất cả luồng hoàn thành
    for f_ in tqdm(futures, desc="Waiting workers"):
        f_.result()
 
print(f"✅ Đã lưu Review tại: {FINAL_REVIEW_CSV}")

🗑️ Đã dọn dẹp file Review cũ. Bắt đầu tạo file mới hoàn toàn...
📦 Đang xử lý Review theo cụm 500,000 (đa luồng)...


Streaming Review: 503662it [00:07, 24413.78it/s] 

A. Chọn cột: giữ lại ['rating', 'title', 'user_id', 'text', 'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
B. Drop NA  : 500,000 → 500,000 (-0)


Streaming Review: 510969it [00:07, 23589.23it/s]

C. Dtype    : 500,000 → 500,000
D. Timestamp: range 2004 - 2023
E. Sentiment: {'positive': 404943, 'negative': 50452, 'neutral': 44605}


...

C. Dtype    : 500,000 → 500,000
D. Timestamp: range 2000 - 2023
E. Sentiment: {'positive': 357537, 'negative': 104034, 'neutral': 38429}


Streaming Review: 66033346it [35:47, 30745.72it/s]
Waiting workers:   0%|          | 0/133 [00:00<?, ?it/s]

A. Chọn cột: giữ lại ['rating', 'title', 'user_id', 'text', 'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
B. Drop NA  : 33,346 → 33,346 (-0)
C. Dtype    : 33,346 → 33,346
D. Timestamp: range 2000 - 2023
E. Sentiment: {'positive': 23759, 'negative': 7050, 'neutral': 2537}
G. Duplicate: 31,350 → 30,906 (-444 duplicates)

✅ Kết quả cuối: 30,906 reviews
G. Duplicate: 470,853 → 454,524 (-16,329 duplicates)

✅ Kết quả cuối: 454,524 reviews


Waiting workers: 100%|██████████| 133/133 [00:04<00:00, 30.56it/s]

✅ Đã lưu Review tại: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\review_clean_all.csv


6. Đọc tiền xử lý + lưu file meta

In [6]:
if os.path.exists(FINAL_META_CSV):
    os.remove(FINAL_META_CSV)
    print("🗑️ Đã xóa file Meta cũ. Bắt đầu tạo file mới...")
def process_and_save_meta(chunk_data, file_exists_flag, write_lock):
    """Preprocess 1 chunk meta rồi ghi CSV — chạy trong luồng con."""
    df_raw  = pd.DataFrame(chunk_data)
    df_temp = preprocess_meta(df_raw)
    del df_raw
    with write_lock:
        df_temp.to_csv(
            FINAL_META_CSV,
            mode='a',
            index=False,
            header=not file_exists_flag[0]
        )
        file_exists_flag[0] = True
    del df_temp
    gc.collect()
 
print(f"📦 Đang xử lý Meta theo cụm {CHUNK_SIZE:,} (đa luồng)...")
 
file_exists_flag = [False]
write_lock       = threading.Lock()
current_chunk    = []
 
with gzip.open(META_PATH, 'rt', encoding='utf-8') as f, \
     ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
 
    futures = []
    for line in tqdm(f, desc="Streaming Meta"):
        try:
            current_chunk.append(json.loads(line.strip()))
        except:
            continue
 
        if len(current_chunk) == CHUNK_SIZE:
            futures.append(executor.submit(
                process_and_save_meta,
                current_chunk.copy(),
                file_exists_flag,
                write_lock
            ))
            current_chunk = []
 
    if current_chunk:
        futures.append(executor.submit(
            process_and_save_meta,
            current_chunk.copy(),
            file_exists_flag,
            write_lock
        ))
 
    for f_ in tqdm(futures, desc="Waiting workers"):
        f_.result()
 
print(f"✅ Đã lưu Meta tại: {FINAL_META_CSV}")

🗑️ Đã xóa file Meta cũ. Bắt đầu tạo file mới...
📦 Đang xử lý Meta theo cụm 500,000 (đa luồng)...


Streaming Meta: 500755it [00:23, 5554.05it/s] 

Raw shape: (500000, 16)
A. Giữ cột: ['parent_asin', 'title', 'description', 'categories', 'average_rating', 'rating_number', 'store', 'main_category', 'images', 'price']
B. Drop NA/dup: 500,000 → 500,000
D. Description: đã chuyển list → string
E. Categories top 5: {'Clothing, Shoes & Jewelry': 223172, 'Shoe, Jewelry & Watch Accessories': 59}


Streaming Meta: 502863it [00:26, 2415.82it/s]


✅ Meta kết quả: (223231, 9)


...

Raw shape: (500000, 16)
A. Giữ cột: ['parent_asin', 'title', 'description', 'categories', 'average_rating', 'rating_number', 'store', 'main_category', 'images', 'price']
B. Drop NA/dup: 500,000 → 500,000
D. Description: đã chuyển list → string
E. Categories top 5: {'Clothing, Shoes & Jewelry': 67183, 'Shoe, Jewelry & Watch Accessories': 14}

✅ Meta kết quả: (67197, 9)


Streaming Meta: 7218481it [06:43, 17908.77it/s]
Waiting workers:   0%|          | 0/15 [00:00<?, ?it/s]

Raw shape: (218481, 16)
A. Giữ cột: ['parent_asin', 'title', 'description', 'categories', 'average_rating', 'rating_number', 'store', 'main_category', 'images', 'price']
B. Drop NA/dup: 218,481 → 218,481
D. Description: đã chuyển list → string
E. Categories top 5: {'Clothing, Shoes & Jewelry': 27112, 'Shoe, Jewelry & Watch Accessories': 6}

✅ Meta kết quả: (27118, 9)


Waiting workers: 100%|██████████| 15/15 [00:02<00:00,  5.02it/s]

✅ Đã lưu Meta tại: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\meta_clean_all.csv


7. Tạo data base của meta data

In [7]:
# DB_PATH = "metadata.db"
# # Xóa db cũ nếu có để làm mới
# if os.path.exists(DB_PATH): os.remove(DB_PATH)

# conn = sqlite3.connect(DB_PATH)

# print("🗄️ Bước 1: Đang chuyển file Meta 12GB vào SQLite...")
# # Đọc Meta theo cụm để không tốn RAM
# meta_reader = pd.read_csv(FINAL_META_CSV, chunksize=200000, low_memory=False)

# for chunk in tqdm(meta_reader, desc="Importing Meta to DB"):
#     # Đưa vào bảng 'products'
#     chunk.to_sql('products', conn, if_exists='append', index=False)

# # TẠO INDEX: Đây là bước quan trọng nhất để tra cứu siêu tốc (O(1))
# print("⚡ Đang tạo Index cho parent_asin (để merge nhanh)...")
# conn.execute("CREATE INDEX idx_asin ON products (parent_asin)")
# conn.close()
# print("✅ Đã tạo xong Database Meta trên ổ cứng!")

8. Merge 2 file bằng hybrid (data base + dictionary)

In [8]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import gc
from tqdm import tqdm

MERGED_OUTPUT_PQ = PROCESSED_DIR / "amazon_full_hybrid_merged.parquet"
CHUNK_SIZE       = 500_000 # Hoặc giảm xuống 200_000 nếu máy tính báo tràn RAM
 
# ------------------------------------------------------------------
# BƯỚC 1: Load Meta + Đánh Index trên parent_asin
# ------------------------------------------------------------------
print("📦 Bước 1: Load Meta và đánh Index...")

# Tùy chọn cột: Nên lọc các cột cần thiết từ Meta để tránh trùng lặp với Review. 
# Giả sử Review đã có: 'user_id', 'parent_asin', 'rating', 'timestamp', 'text'
# Ta chỉ lấy các cột bổ sung từ Meta:
META_COLS_TO_KEEP = ['parent_asin', 'title', 'categories', 'average_rating', 'description']

df_meta = pd.read_csv(FINAL_META_CSV, usecols=META_COLS_TO_KEEP, low_memory=False)
df_meta = df_meta.set_index('parent_asin')
 
print(f"   ✅ Meta shape : {df_meta.shape}")
print(f"   ✅ Index dtype: {df_meta.index.dtype}")
 
# ------------------------------------------------------------------
# BƯỚC 2: Đọc Review theo chunk + Merge tuần tự
# ------------------------------------------------------------------
print(f"\n🚀 Bước 2: Merge theo chunk {CHUNK_SIZE:,} (tuần tự + index)...")
 
review_reader = pd.read_csv(FINAL_REVIEW_CSV, chunksize=CHUNK_SIZE, low_memory=False)
pq_writer = None
base_schema = None # Lưu lại cấu trúc chuẩn của chunk đầu tiên
 
for i, chunk in enumerate(tqdm(review_reader, desc="Merging")):
    
    # how='left': giữ tất cả review dù không có meta
    merged = chunk.join(df_meta, on='parent_asin', how='left', rsuffix='_meta')
 
    # Chuyển đổi sang PyArrow
    table = pa.Table.from_pandas(merged, preserve_index=False)
    
    if pq_writer is None:
        # Lấy schema của chunk đầu tiên làm chuẩn mực
        base_schema = table.schema
        pq_writer = pq.ParquetWriter(MERGED_OUTPUT_PQ, base_schema)
    else:
        # ÉP KIỂU: Bắt buộc các chunk tiếp theo phải uốn theo khuôn của chunk 1
        # Giúp triệt tiêu hoàn toàn lỗi văng code do sai lệch datatype
        table = table.cast(base_schema)
        
    # Ghi ra file
    pq_writer.write_table(table)
 
    # Giải phóng RAM ngay lập tức
    del chunk, merged, table
    gc.collect()
 
if pq_writer:
    pq_writer.close()
 
# Dọn dẹp nốt file Meta khi đã hoàn tất
del df_meta
gc.collect()

print(f"\n✨ HOÀN THÀNH! File merged: {MERGED_OUTPUT_PQ}")

📦 Bước 1: Load Meta và đánh Index...
   ✅ Meta shape : (1459771, 4)
   ✅ Index dtype: object

🚀 Bước 2: Merge theo chunk 500,000 (tuần tự + index)...


Merging: 125it [08:37,  4.14s/it]



✨ HOÀN THÀNH! File merged: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\amazon_full_hybrid_merged.parquet


<span style = "font-size : 30px" >9. Cold start + lưu <span>

In [9]:
import pyarrow as pa
import pyarrow.parquet as pq
import gc
from concurrent.futures import ThreadPoolExecutor

INPUT_PARQUET   = PROCESSED_DIR / "amazon_full_hybrid_merged.parquet"
OUTPUT_FINAL_PQ = PROCESSED_DIR / "amazon_clothing_final_gold_daluong.parquet"
K_CORE    = 5
N_WORKERS = 4

# =============================================================================
# BƯỚC 1: TÍNH K-CORE (tuần tự)
# =============================================================================
print(f"🔍 Bắt đầu lọc Cold-start (k={K_CORE})...")
df_ids = pd.read_parquet(INPUT_PARQUET, columns=['user_id', 'parent_asin'])

iteration = 0
while True:
    iteration += 1
    n_before = len(df_ids)

    item_counts = df_ids['parent_asin'].value_counts()
    valid_items = item_counts[item_counts >= K_CORE].index
    df_ids = df_ids[df_ids['parent_asin'].isin(valid_items)]

    user_counts = df_ids['user_id'].value_counts()
    valid_users = user_counts[user_counts >= K_CORE].index
    df_ids = df_ids[df_ids['user_id'].isin(valid_users)]

    n_after = len(df_ids)
    print(f"🔄 Vòng lặp {iteration}: {n_before:,} -> {n_after:,} dòng")

    if n_before == n_after:
        print("✅ K-Core hội tụ.")
        break

valid_user_set = set(df_ids['user_id'].unique())
valid_item_set = set(df_ids['parent_asin'].unique())
del df_ids
gc.collect()

# =============================================================================
# BƯỚC 2: ĐỌC + LỌC SONG SONG, GHI TUẦN TỰ
# =============================================================================
parquet_file = pq.ParquetFile(INPUT_PARQUET)
num_rg       = parquet_file.num_row_groups
print(f"\n💾 Xử lý {num_rg} row groups với {N_WORKERS} luồng...")

def process_row_group(rg_index):
    """Đọc + lọc 1 row group, giải phóng RAM ngay trong luồng con."""
    chunk = parquet_file.read_row_group(rg_index).to_pandas()
    filtered = chunk[
        chunk['user_id'].isin(valid_user_set) &
        chunk['parent_asin'].isin(valid_item_set)
    ]
    del chunk  # Giải phóng chunk gốc ngay

    if filtered.empty:
        del filtered
        gc.collect()  # Dọn ngay trong luồng con, không chờ GIL
        return None

    table = pa.Table.from_pandas(filtered, preserve_index=False)
    del filtered
    gc.collect()  # Dọn ngay sau khi đã convert sang Arrow
    return table

pq_writer = None

with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    # executor.map: xử lý song song, trả kết quả đúng thứ tự
    for table in tqdm(
        executor.map(process_row_group, range(num_rg)),
        total=num_rg,
        desc="Processing"
    ):
        if table is not None:
            if pq_writer is None:
                pq_writer = pq.ParquetWriter(OUTPUT_FINAL_PQ, table.schema)
            pq_writer.write_table(table)
            del table
            gc.collect()  # Dọn sau khi ghi xong

if pq_writer:
    pq_writer.close()

print(f"\n✨ HOÀN THÀNH! Gold Dataset: {OUTPUT_FINAL_PQ}")

🔍 Bắt đầu lọc Cold-start (k=5)...
🔄 Vòng lặp 1: 62,381,693 -> 24,052,579 dòng
🔄 Vòng lặp 2: 24,052,579 -> 21,693,321 dòng
🔄 Vòng lặp 3: 21,693,321 -> 21,529,341 dòng
🔄 Vòng lặp 4: 21,529,341 -> 21,516,339 dòng
🔄 Vòng lặp 5: 21,516,339 -> 21,515,311 dòng
🔄 Vòng lặp 6: 21,515,311 -> 21,515,263 dòng
🔄 Vòng lặp 7: 21,515,263 -> 21,515,259 dòng
🔄 Vòng lặp 8: 21,515,259 -> 21,515,259 dòng
✅ K-Core hội tụ.

💾 Xử lý 125 row groups với 4 luồng...


Processing: 100%|██████████| 125/125 [11:22<00:00,  5.46s/it]


✨ HOÀN THÀNH! Gold Dataset: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\amazon_clothing_final_gold_daluong.parquet
